# Pick the best checkpoint and promote it for inference

Workflow for comparing the different trained runs (on the **server**), bringing the
winner down to your **local** machine, and promoting it into `checkpoints/pretrained/`
(the only checkpoint folder that is git-tracked).

**Where each step runs:**

| Step | Runs on | What it does |
|------|---------|--------------|
| 1 | server | Rank every run by best `val_loss` |
| 2 | server | Held-out eval of the top candidates (the real test) |
| 3 | local  | `scp` just the winner down |
| 4 | local  | Promote winner into `checkpoints/pretrained/`, then commit + push |
| 5 | local  | Run inference with the promoted weights |

Run all cells from the **project root** (`data_interpolation/`), not from `notebooks/`.

In [ ]:
# Run once: move from notebooks/ up to the project root so relative paths work.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

## Step 1 — (server) Rank all runs by validation loss

Scans every `checkpoints/*/history.json` and prints the best (lowest) `val_loss`
per run, sorted. Top line = best by training metric.

In [ ]:
import json, glob, os

rows = []
for h in glob.glob("checkpoints/*/history.json"):
    run = os.path.basename(os.path.dirname(h))
    try:
        hist = json.load(open(h))
        vals = [e["val_loss"] for e in hist if e.get("val_loss") is not None]
        if vals:
            best = min(vals)
            last = hist[-1]
            rows.append((best, run, last.get("epoch"), last.get("train_loss")))
    except Exception as e:
        print(f"  ! skip {run}: {e}")

print(f"{'best_val_loss':>14}  {'run':<28} {'last_ep':>7} {'train_loss':>11}")
for best, run, ep, tr in sorted(rows):
    tr_s = tr if tr is None else f"{tr:.5f}"
    print(f"{best:>14.5f}  {run:<28} {ep!s:>7} {tr_s:>11}")

## Step 2 — (server) Confirm with held-out eval

`val_loss` can mislead if runs used different normalization or residual settings.
Validate the top 2–3 candidates on a real held-out BOLD file with `eval.py`.
Compare `mean_model_psnr` (higher = better) and `model_beats_naive_pct`.

⚠️ **Add `--residual` for any run trained with it** — otherwise its scores are
meaningless. Check each run's README/config.

Edit `CANDIDATES` and `HELD_OUT_FILE` below, then run.

In [ ]:
import subprocess

CANDIDATES = ["run_100ep", "timing_2epoch_pretrained_v3"]  # <-- edit: top runs from Step 1
HELD_OUT_FILE = "data/<held_out_bold>.nii.gz"             # <-- edit: a file NOT used in training
RESIDUAL_RUNS = set()                                      # <-- add run names trained with --residual

for run in CANDIDATES:
    cmd = [
        "python", "eval.py",
        "--weights", f"checkpoints/{run}/model_weights.pt",
        "--file", HELD_OUT_FILE,
        "--output-dir", f"results/eval_{run}",
        "--history", f"checkpoints/{run}/history.json",
    ]
    if run in RESIDUAL_RUNS:
        cmd.append("--residual")
    print(f"\n=== Evaluating: {run} ===")
    subprocess.run(cmd, check=True)

## Step 3 — (local) Bring just the winner down

Run this on your **local machine**. Edit the host and remote path. This copies only
the two files you need into `/tmp/` (we don't pollute the local `checkpoints/`).

In [ ]:
WINNER = "run_100ep"                                                       # <-- edit
SSH = "user@server"                                                        # <-- edit
REMOTE = "/path/to/CAI-MedImg/data_interpolation"                          # <-- edit

import subprocess
for fname in ("model_weights.pt", "history.json"):
    src = f"{SSH}:{REMOTE}/checkpoints/{WINNER}/{fname}"
    dst = f"/tmp/winner_{fname}"
    print(f"scp {src} -> {dst}")
    subprocess.run(["scp", src, dst], check=True)
print("done")

## Step 4 — (local) Promote into `pretrained/` and commit

Overwrites the tracked `checkpoints/pretrained/` weights + history with the winner.
No `.gitignore` change needed — `pretrained/` is already whitelisted.

**Update `checkpoints/pretrained/README.md` by hand** after this if the winner's
architecture, normalization, residual flag, or training schedule differs from the
current one.

In [ ]:
import shutil
shutil.copy("/tmp/winner_model_weights.pt", "checkpoints/pretrained/model_weights.pt")
shutil.copy("/tmp/winner_history.json",     "checkpoints/pretrained/history.json")
print("promoted winner into checkpoints/pretrained/")

In [ ]:
# Commit + push (run from project root). Review the diff before running.
!git add checkpoints/pretrained/model_weights.pt checkpoints/pretrained/history.json checkpoints/pretrained/README.md
!git status
# !git commit -m "Promote <winner> checkpoint to pretrained"
# !git push

## Step 5 — (local) Run inference with the promoted weights

Add `--residual` here **only** if the winner was trained with it.

In [ ]:
!python main.py \
    --weights checkpoints/pretrained/model_weights.pt \
    --input  data/your_bold.nii.gz \
    --output results/your_bold_2x.nii.gz